In [10]:
import requests
import json
from datetime import datetime

# API Endpoint JSON Publik BMKG
BASE_URL = "https://api.bmkg.go.id/publik/prakiraan-cuaca?adm4="

# Dictionary Kota Target menggunakan referensi kode wilayah (adm4)
# Kode ini mewakili titik kelurahan/kecamatan di kota-kota strategis
target_cities = {
    "Banda Aceh": "11.71.03.2001",   # Representasi area Syiah Kuala / Kopelma
    "Jakarta Selatan": "31.74.08.1004", # Representasi area Setiabudi/Kuningan
    "Medan": "12.71.06.1001"         # Representasi area Petisah Tengah
}

print("Engine MosqRisk Multi-City diinisialisasi...")

Engine MosqRisk Multi-City diinisialisasi...


In [11]:
def calculate_mosquito_risk(temp, humidity):
    score = 20 # Base score
    # Logika Suhu
    if temp is not None:
        if 26 <= temp <= 30: score += 40
        elif 24 <= temp < 26 or 30 < temp <= 32: score += 20
    # Logika Kelembapan
    if humidity is not None:
        if humidity > 80: score += 40
        elif 75 <= humidity <= 80: score += 25
    
    score = min(score, 100)
    category = "TINGGI" if score >= 75 else "SEDANG" if score >= 50 else "RENDAH"
    return score, category

def fetch_city_weather(adm4_code):
    try:
        # Header standar yang bersih karena endpoint ini memang ditujukan untuk publik/developer
        headers = {'User-Agent': 'MosqRisk-Engine/1.0'}
        response = requests.get(f"{BASE_URL}{adm4_code}", headers=headers, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            # Menavigasi struktur JSON BMKG untuk mengambil data cuaca saat ini
            # Menggunakan .get() agar terhindar dari KeyError jika struktur berubah
            try:
                # Biasanya data cuaca ada di indeks pertama array
                current_weather = data['data'][0]['cuaca'][0][0]
                temp = float(current_weather.get('t', 0))
                humidity = float(current_weather.get('hu', 0))
                return temp, humidity
            except (KeyError, IndexError, TypeError):
                print(f"Struktur JSON tidak terduga untuk kode {adm4_code}.")
                return None, None
        else:
            print(f"Gagal akses API untuk kode {adm4_code}. Status: {response.status_code}")
            return None, None
    except Exception as e:
        print(f"Error koneksi ke BMKG: {e}")
        return None, None

In [12]:
# Proses Eksekusi Penarikan Data Serentak
final_output = []
print("Mulai menarik data cuaca real-time dari BMKG...\n")

for city_name, adm4 in target_cities.items():
    print(f"-> Memproses data wilayah: {city_name} (Kode: {adm4})...")
    temp, humidity = fetch_city_weather(adm4)
    
    if temp and humidity:
        risk_score, risk_status = calculate_mosquito_risk(temp, humidity)
        
        city_data = {
            "location": city_name,
            "temperature_celsius": temp,
            "humidity_percent": humidity,
            "risk_score": risk_score,
            "risk_status": risk_status,
            "last_updated": datetime.now().isoformat()
        }
        final_output.append(city_data)
    else:
        print(f"    [!] Gagal memproses {city_name}")

print("\n=== OUTPUT JSON FINAL UNTUK FRONTEND NEXT.JS ===")
print(json.dumps(final_output, indent=4))

Mulai menarik data cuaca real-time dari BMKG...

-> Memproses data wilayah: Banda Aceh (Kode: 11.71.03.2001)...
-> Memproses data wilayah: Jakarta Selatan (Kode: 31.74.08.1004)...
-> Memproses data wilayah: Medan (Kode: 12.71.06.1001)...

=== OUTPUT JSON FINAL UNTUK FRONTEND NEXT.JS ===
[
    {
        "location": "Banda Aceh",
        "temperature_celsius": 26.0,
        "humidity_percent": 84.0,
        "risk_score": 100,
        "risk_status": "TINGGI",
        "last_updated": "2026-06-29T02:44:53.622385"
    },
    {
        "location": "Jakarta Selatan",
        "temperature_celsius": 25.0,
        "humidity_percent": 89.0,
        "risk_score": 80,
        "risk_status": "TINGGI",
        "last_updated": "2026-06-29T02:44:53.711790"
    },
    {
        "location": "Medan",
        "temperature_celsius": 24.0,
        "humidity_percent": 96.0,
        "risk_score": 80,
        "risk_status": "TINGGI",
        "last_updated": "2026-06-29T02:44:53.809543"
    }
]
